# Store Clustering EDA

In this notebook we analyze the relationship between store clusters, store types, and the distribution of sales among product families. We explore:
- Sales patterns by cluster
- Sales patterns by store type
- Family sales distribution within clusters
- Geographic patterns (state, city) and their relationship to clusters
- Statistical differences between clusters

In [23]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from scipy import stats


In [14]:
# Load data
df_train = pd.read_csv('../../data/train.csv')
df_stores = pd.read_csv('../../data/stores.csv')

# Convert date to datetime
df_train['date'] = pd.to_datetime(df_train['date'])

print(f"Training data shape: {df_train.shape}")
print(f"Stores data shape: {df_stores.shape}")
print(f"\nStores info:")
print(df_stores.head())


Training data shape: (3000888, 6)
Stores data shape: (54, 5)

Stores info:
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4


In [15]:
# Merge train data with store information
df_train_store = df_train.merge(df_stores, on='store_nbr', how='left')

print("Merged data shape:", df_train_store.shape)
print("\nStore characteristics:")
print(f"Number of unique stores: {df_train_store['store_nbr'].nunique()}")
print(f"Number of clusters: {df_train_store['cluster'].nunique()}")
print(f"Store types: {df_train_store['type'].unique()}")
print(f"Number of states: {df_train_store['state'].nunique()}")
print(f"Number of cities: {df_train_store['city'].nunique()}")


Merged data shape: (3000888, 10)

Store characteristics:
Number of unique stores: 54
Number of clusters: 17
Store types: ['D' 'C' 'B' 'E' 'A']
Number of states: 16
Number of cities: 22


## Cluster Overview

In [16]:
# Cluster distribution
cluster_info = df_stores.groupby('cluster').agg({
    'store_nbr': 'count',
    'type': lambda x: x.value_counts().to_dict(),
    'state': lambda x: x.nunique(),
    'city': lambda x: x.nunique()
}).reset_index()
cluster_info.columns = ['cluster', 'num_stores', 'type_distribution', 'num_states', 'num_cities']

print("Cluster Overview:")
print(cluster_info)


Cluster Overview:
    cluster  num_stores         type_distribution  num_states  num_cities
0         1           3                  {'D': 3}           2           3
1         2           2                  {'D': 2}           1           1
2         3           7                  {'C': 7}           5           6
3         4           3                  {'D': 3}           3           3
4         5           1                  {'A': 1}           1           1
5         6           6                  {'B': 6}           4           5
6         7           2                  {'C': 2}           2           2
7         8           3                  {'D': 3}           1           1
8         9           2                  {'D': 2}           2           2
9        10           6  {'E': 4, 'D': 1, 'B': 1}           3           4
10       11           3                  {'A': 3}           2           2
11       12           1                  {'C': 1}           1           1
12       13         

In [17]:
# Cross-tabulation of store type and cluster
crosstab = pd.crosstab(df_stores['type'], df_stores['cluster'])
print("Store Type vs Cluster:")
print(crosstab)
print("\nWe can see that each cluster only has one type of stores except cluster 10")


Store Type vs Cluster:
cluster  1   2   3   4   5   6   7   8   9   10  11  12  13  14  15  16  17
type                                                                       
A         0   0   0   0   1   0   0   0   0   0   3   0   0   4   0   0   1
B         0   0   0   0   0   6   0   0   0   1   0   0   0   0   0   1   0
C         0   0   7   0   0   0   2   0   0   0   0   1   0   0   5   0   0
D         3   2   0   3   0   0   0   3   2   1   0   0   4   0   0   0   0
E         0   0   0   0   0   0   0   0   0   4   0   0   0   0   0   0   0

We can see that each cluster only has one type of stores except cluster 10


In [18]:
# Visualize cluster distribution
fig = px.bar(cluster_info, x='cluster', y='num_stores', 
             title='Number of Stores per Cluster',
             labels={'cluster': 'Cluster', 'num_stores': 'Number of Stores'})
fig.show()


## Sales Analysis by Cluster


In [19]:
# Sales statistics by cluster
sales_by_cluster = df_train_store.groupby('cluster')['sales'].agg([
    'mean', 'median', 'std', 'sum', 'count', 'min', 'max'
]).sort_values('mean', ascending=False)

print("Sales Statistics by Cluster:")
print(sales_by_cluster)


Sales Statistics by Cluster:
                mean  median          std           sum   count  min  \
cluster                                                                
5        1117.245254    75.0  2685.282436  6.208755e+07   55572  0.0   
14        708.227718    36.0  1804.016195  1.574305e+08  222288  0.0   
8         647.377856    42.0  1595.398546  1.079282e+08  166716  0.0   
11        603.507018     5.0  1823.033441  1.006143e+08  166716  0.0   
17        592.231511    27.0  1352.209195  3.291149e+07   55572  0.0   
6         342.661732     7.0  1045.356963  1.142544e+08  333432  0.0   
1         326.163967    15.0   825.005416  5.437675e+07  166716  0.0   
12        324.461406    10.0   852.435149  1.803097e+07   55572  0.0   
13        324.364108    14.0   876.439838  7.210225e+07  222288  0.0   
4         296.572872    19.0   725.281732  4.944344e+07  166716  0.0   
9         274.968339    16.0   655.577614  3.056108e+07  111144  0.0   
2         260.170621     5.0   732.

In [20]:
# Visualize average sales by cluster
fig = px.bar(x=sales_by_cluster.index.astype(str), 
             y=sales_by_cluster['mean'].values, 
             title='Average Sales by Cluster',
             labels={'x': 'Cluster', 'y': 'Average Sales'})
fig.update_layout(xaxis_title='Cluster', yaxis_title='Average Sales')
fig.show()


In [21]:
# Total sales by cluster
fig = px.bar(x=sales_by_cluster.index.astype(str), 
             y=sales_by_cluster['sum'].values, 
             title='Total Sales by Cluster',
             labels={'x': 'Cluster', 'y': 'Total Sales'})
fig.update_layout(xaxis_title='Cluster', yaxis_title='Total Sales')
fig.show()


## Sales Analysis by Store Type


In [22]:
# Sales statistics by store type
sales_by_type = df_train_store.groupby('type')['sales'].agg([
    'mean', 'median', 'std', 'sum', 'count', 'min', 'max'
]).sort_values('mean', ascending=False)

print("Sales Statistics by Store Type:")
print(sales_by_type)


Sales Statistics by Store Type:
            mean  median          std           sum    count  min         max
type                                                                         
A     705.878743    24.0  1892.700760  3.530438e+08   500148  0.0   76090.000
D     350.979407    16.0   965.728732  3.510833e+08  1000296  0.0  124717.000
B     326.739714     7.0   977.528999  1.452606e+08   444576  0.0   89576.360
E     269.121301     4.0   761.422519  5.982244e+07   222288  0.0   16542.902
C     197.263301     5.0   581.310901  1.644347e+08   833580  0.0   45361.000


In [11]:
# Visualize average sales by store type
fig = px.bar(x=sales_by_type.index, y=sales_by_type['mean'].values, 
             title='Average Sales by Store Type',
             labels={'x': 'Store Type', 'y': 'Average Sales'})
fig.show()
